In [1]:
import os
import numpy as np
from flask import Flask, render_template, request, redirect, url_for, flash
from werkzeug.utils import secure_filename

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.imagenet_utils import preprocess_input

app = Flask(__name__)
app.secret_key = "change_this_secret_key"

# ---------------- Config ----------------
UPLOAD_FOLDER = "static/uploads"
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "bmp", "webp"}

app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# ---------------- Labels ----------------
labels_map = {'A+': 0, 'A-': 1, 'AB+': 2, 'AB-': 3, 'B+': 4, 'B-': 5, 'O+': 6, 'O-': 7}
idx_to_label = {v: k for k, v in labels_map.items()}

# ---------------- Model registry ----------------
# You can tweak target_size per model if needed.
MODEL_DIR = "model_weights"
MODELS = {
    "resnet": {
        "display": "ResNet",
        "path": os.path.join(MODEL_DIR, "Resnet_blood_group_detection.h5"),
        "target_size": (256, 256),
        "preprocess": "imagenet",  # uses preprocess_input
    },
    "lenet": {
        "display": "LeNet",
        "path": os.path.join(MODEL_DIR, "model_blood_group_detection_lenet.keras"),
        "target_size": (224, 224),
        "preprocess": "scale01",   # x/255
    },
    "alexnet": {
        "display": "AlexNet",
        "path": os.path.join(MODEL_DIR, "model_blood_group_detection_alextnet.keras"),
        "target_size": (256, 256),
        "preprocess": "imagenet",
    },
    "vgg16": {
        "display": "VGG16",
        "path": os.path.join(MODEL_DIR, "blood_group_detection.h5"),
        "target_size": (256, 256),
        "preprocess": "imagenet",
    },
}

# Cache loaded models so they load once
MODEL_CACHE = {}


def allowed_file(filename: str) -> bool:
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS


def get_model(model_key: str):
    """Load model once and reuse."""
    if model_key not in MODEL_CACHE:
        model_path = MODELS[model_key]["path"]
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found: {model_path}")
        MODEL_CACHE[model_key] = load_model(model_path)
    return MODEL_CACHE[model_key]


def prepare_image(img_path: str, target_size=(256, 256), preprocess_type="imagenet"):
    """Prepare image array for prediction depending on model preprocessing."""
    img = image.load_img(img_path, target_size=target_size)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)

    if preprocess_type == "imagenet":
        x = preprocess_input(x)
    elif preprocess_type == "scale01":
        x = x / 255.0
    else:
        # fallback (no preprocess)
        x = x.astype("float32")

    return x


def predict_with_model(model_key: str, img_path: str):
    """Return label + confidence% + raw probs (optional)."""
    cfg = MODELS[model_key]
    model = get_model(model_key)

    x = prepare_image(
        img_path,
        target_size=cfg["target_size"],
        preprocess_type=cfg["preprocess"],
    )

    preds = model.predict(x, verbose=0)
    preds = np.array(preds)

    # handle shape (1, num_classes)
    if preds.ndim == 2:
        probs = preds[0]
    else:
        probs = preds.reshape(-1)

    idx = int(np.argmax(probs))
    label = idx_to_label.get(idx, str(idx))
    confidence = float(probs[idx]) * 100.0

    return label, confidence


@app.route("/", methods=["GET"])
def home():
    # for dropdown
    model_options = [{"key": "all", "name": "All Models"}] + [
        {"key": k, "name": MODELS[k]["display"]} for k in MODELS
    ]
    return render_template("index.html", model_options=model_options)


@app.route("/predict", methods=["POST"])
def predict():
    if "file" not in request.files:
        flash("No file part in request.")
        return redirect(url_for("home"))

    file = request.files["file"]
    selected = request.form.get("model_choice", "all")

    if file.filename == "":
        flash("No file selected.")
        return redirect(url_for("home"))

    if not allowed_file(file.filename):
        flash("Invalid file type. Please upload png/jpg/jpeg/bmp/webp.")
        return redirect(url_for("home"))

    filename = secure_filename(file.filename)
    save_path = os.path.join(app.config["UPLOAD_FOLDER"], filename)
    file.save(save_path)

    image_url = url_for("static", filename=f"uploads/{filename}")

    # Build model options again (for re-render with selection kept)
    model_options = [{"key": "all", "name": "All Models"}] + [
        {"key": k, "name": MODELS[k]["display"]} for k in MODELS
    ]

    results = []
    try:
        if selected == "all":
            for mk in MODELS:
                label, conf = predict_with_model(mk, save_path)
                results.append({
                    "model_key": mk,
                    "model_name": MODELS[mk]["display"],
                    "label": label,
                    "confidence": f"{conf:.2f}"
                })
        else:
            if selected not in MODELS:
                flash("Invalid model selection.")
                return redirect(url_for("home"))

            label, conf = predict_with_model(selected, save_path)
            results.append({
                "model_key": selected,
                "model_name": MODELS[selected]["display"],
                "label": label,
                "confidence": f"{conf:.2f}"
            })

    except Exception as e:
        flash(f"Prediction failed: {e}")
        return redirect(url_for("home"))

    return render_template(
        "index.html",
        model_options=model_options,
        selected_model=selected,
        image_url=image_url,
        results=results,
    )


if __name__ == "__main__":
    app.run(debug=False,host="0.0.0.0")


ImportError: Traceback (most recent call last):
  File "C:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 73, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [2]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
print(np.__version__)

2.4.0


In [4]:
!pip install flask

In [2]:
%pip install flask werkzeug tensorflow

  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached werkzeug-3.1.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.2-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached termcolor-3.2.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached tensorboard-2.20.0-py3-none-any.wh